#**2. VLM Semantic Description of Retro-Futuristic Images**

*Creates semantic description file, detailing visual content of each image  **semantic_descriptions.json***

This notebook executes the automated, vision-language processing phase for the **QRC-Mosaic** pipeline. It leverages the state-of-the-art **Qwen2.5-VL-7B-Instruct** Vision-Language Model (VLM) to generate objective, un-biased semantic descriptions and controlled thematic classifications for a historical corpus of retro-futuristic artworks.

Through a structured 11-stage pipeline, the notebook enables users to:
1. **Initialise Compute & Environment:** Detect and configure GPU hardware (NVIDIA T4/CUDA) and install required dependencies (`transformers`, `accelerate`, `qwen-vl-utils`).
2. **Access Data Corpus & Validate Assets:** Mount Google Drive and verify the integrity of the 64 original raw artwork assets and sequential IDs (`FV001`–`FV064`).
3. **Deploy & Configure VLM Architecture:** Instantiate `Qwen2.5-VL-7B-Instruct` in FP16 precision with auto-device mapping and custom prompt constraints to prevent subjective interpretation.
4. **Execute Resumable Batch Processing:** Run memory-efficient, single-image inference across the 64-asset corpus, incorporating automated JSON parsing, controlled topic normalisation, and real-time checkpointing.
5. **Handle Exceptions & Export Registry:** Catch edge-case classification exceptions (e.g. `FV038`), apply normalisation rules, and export the final verified dataset to `semantic_descriptions.json`.

This structured semantic metadata provides the qualitative foundation required for subsequent quantum sorting and mosaic assembly algorithms.

### 1. Hardware Detection and CUDA Environment Audit

Detects available hardware acceleration (NVIDIA GPU) and verifies PyTorch and CUDA runtime environments.

**Features:**
- **PyTorch Verification:** Displays installed PyTorch framework version.
- **CUDA Runtime Audit:** Checks for active CUDA availability to ensure GPU-accelerated inference.
- **Hardware Identifier:** Identifies and logs the active GPU device name (e.g. Tesla T4).

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


### 2. Dependency Installation and Environment Setup

Installs essential Python packages required for Hugging Face model loading and Vision-Language processing.

**Features:**
- **Framework Loading:** Installs updated versions of `transformers` and `accelerate`.
- **VLM Utilities:** Installs `qwen-vl-utils` for processing visual messages and image inputs.

In [ ]:
!pip install -q -U transformers accelerate qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 21.9 MB/s eta 0:00:00


### 3. Storage Mounting and Cloud Directory Access

Mounts Google Drive to provide persistent storage access for input images and output metadata registries.

**Features:**
- **Cloud Connection:** Establishes secure access to `/content/drive`.
- **Persistent Storage:** Ensures output JSON registries persist across session disconnections.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


### 4. Image Corpus Integrity and ID Validation

Locates the artwork directory and performs an integrity check across the 64-asset historical image corpus.

**Features:**
- **Corpus Ingestion:** Scans the target directory for standard image formats (`.jpg`, `.jpeg`, `.png`, `.webp`).
- **ID Extraction:** Regex-parses sequential identifiers (`FV001`–`FV064`) from artwork filenames.
- **Integrity Verification:** Checks for missing files or duplicate IDs, ensuring exactly 64 unique assets are ready.

In [ ]:
from google.colab import drive
from pathlib import Path
import re

MOTH_DIR = Path("/content/drive/MyDrive/MOTH Hackathon 2026 Tools/original")

# verify the image count and duplicate IDs
import re
from collections import Counter

ORIGINAL_DIR = Path(
    "/content/drive/MyDrive/MOTH Hackathon 2026 Tools/original"
)

original_files = sorted(
    p for p in ORIGINAL_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
)

image_ids = [
    re.match(r"^(FV\d{3})_", p.name).group(1)
    for p in original_files
    if re.match(r"^(FV\d{3})_", p.name)
]

print("Image files:", len(original_files))
print("Unique image IDs:", len(set(image_ids)))
print("Duplicate IDs:", len(image_ids) - len(set(image_ids)))

if len(original_files) == 64 and len(set(image_ids)) == 64:
    print("✓ Ready: 64 unique original images")
else:
    print("⚠ Check image set")

Image files: 64
Unique image IDs: 64
Duplicate IDs: 0
✓ Ready: 64 unique original images


### 5. VLM Model Instantiation and Initialization

Downloads and loads the `Qwen2.5-VL-7B-Instruct` model and processor from Hugging Face into GPU memory.

**Features:**
- **Precision Loading:** Loads model weights in `torch.float16` precision for optimal memory efficiency on T4 GPUs.
- **Automated Mapping:** Utilises `device_map="auto"` for balanced memory distribution.
- **Processor Setup:** Initialises the matching `AutoProcessor` for multimodal chat templates and vision tokenization.

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

print("✓ Model loaded")
print("Device:", model.device)

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✓ Model loaded
Device: cuda:0


### 6. Single Asset Ingestion Test

Tests image loading and RGB conversion on a single representative asset (`FV001_grand_tour.jpg`).

**Features:**
- **File Access:** Loads target file from Google Drive via `pathlib.Path`.
- **Format Normalisation:** Converts image pixels to standard 3-channel RGB format and logs original image dimensions.

In [ ]:
import json
from PIL import Image

# Select the first original image
test_path = ORIGINAL_DIR / "FV001_grand_tour.jpg"

image = Image.open(test_path).convert("RGB")

print("Image:", test_path.name)
print("Size:", image.size)

Image: FV001_grand_tour.jpg
Size: (2607, 3886)


### 7. Memory-Efficient Aspect Ratio Resizing

Resizes high-resolution artwork to a maximum dimension of 1024px while preserving complete aspect ratio.

**Features:**
- **Lanczos Resampling:** Applies high-quality Lanczos resampling to reduce image size without cropping visual content.
- **VRAM Optimisation:** Limits visual token consumption to prevent GPU out-of-memory (OOM) errors during inference.

In [ ]:
# Prepare a memory-efficient full-image test
from PIL import Image

TEST_FILE = ORIGINAL_DIR / "FV001_grand_tour.jpg"

image = Image.open(TEST_FILE).convert("RGB")

# Keep the complete image, but reduce its maximum dimension
MAX_IMAGE_SIZE = 1024

image.thumbnail((MAX_IMAGE_SIZE, MAX_IMAGE_SIZE), Image.Resampling.LANCZOS)

print("Resized image:", image.size)

Resized image: (687, 1024)


### 8. VLM Prompt Design and Single-Asset Test Execution

Executes end-to-end VLM inference on asset `FV001` using a constrained, objective vision prompt and JSON formatting instructions.

**Features:**
- **Constrained Prompting:** Enforces strict visual-only description rules, prohibiting subjective interpretation, date guessing, or artist identification.
- **Controlled Vocabularies:** Mandates exact primary and secondary topic classification from a 13-category taxonomy.
- **Chat Template Formatting:** Constructs multimodal input payloads (`image` + `text`) and applies model chat templates.
- **Greedy Inference:** Executes `model.generate()` with greedy decoding (`do_sample=False`) for deterministic, reproducible outputs.

In [ ]:
import json

VLM_PROMPT = """
You are analysing one image from a historical collection of imagined future environments and technologies.

Describe ONLY what is visibly depicted in the image.

Do not infer or state:
- the date of the image
- the creator or artist
- the source or collection
- historical or cultural context
- the artist's intention or motivation
- the meaning or symbolism of the image
- technological feasibility
- whether the depicted technology actually existed
- information from the filename, metadata, or external knowledge

Do not identify the artwork by its title or by recognising it from prior knowledge.
Ignore readable titles, captions, labels, logos, or other text when describing the scene.

Focus on observable visual content:
- the overall scene or setting
- major visible objects and structures
- vehicles or spacecraft
- buildings, habitats or infrastructure
- people or human activity
- landscapes or celestial environments
- machines, equipment or technological systems
- transportation, communication or energy-related elements
- other visually prominent features

If an object cannot be identified confidently, describe its visible form using a general visual term rather than guessing its identity.

Produce a concise description of approximately 30–60 words.
Assign topics using this controlled vocabulary.

Primary topic must be exactly ONE of:
spacecraft
space station
planetary surface
extraterrestrial habitat
city / settlement
transportation
astronomy
communications
energy / infrastructure
robotics / automation
human activity
futuristic technology
other

Secondary topics may contain ZERO OR MORE of the same categories.

Use only direct visual observations. Do not use phrases such as "suggesting", "appears to represent", "evokes", "conveys", or other interpretive language.

Describe visual features directly rather than describing their effect on the viewer. For example, state "overlapping lines and shapes" rather than "creating a sense of motion or depth".

Return ONLY valid JSON with these fields:
{
  "description": "...",
  "primary_topic": "...",
  "secondary_topics": ["..."],
  "visible_elements": ["..."]
}
"""

# ---------------------------------------------------------
# Prepare the test image
# ---------------------------------------------------------

TEST_FILE = ORIGINAL_DIR / "FV001_grand_tour.jpg"

image = Image.open(TEST_FILE).convert("RGB")

# Resize the complete image without cropping.
# This reduces visual-token and GPU-memory requirements.
MAX_IMAGE_SIZE = 1024
image.thumbnail(
    (MAX_IMAGE_SIZE, MAX_IMAGE_SIZE),
    Image.Resampling.LANCZOS
)

print("Image:", TEST_FILE.name)
print("Image size:", image.size)

# ---------------------------------------------------------
# Build the Qwen-VL message
# ---------------------------------------------------------

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": VLM_PROMPT},
        ],
    }
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# ---------------------------------------------------------
# Process image + prompt
# ---------------------------------------------------------

inputs = processor(
    text=[text],
    images=[image],
    padding=True,
    return_tensors="pt",
)

# The model was loaded with device_map="auto", so some
# model components may be offloaded from the GPU.

inputs = {
    k: v.to(model.device) if hasattr(v, "to") else v
    for k, v in inputs.items()
}

# ---------------------------------------------------------
# Generate description
# ---------------------------------------------------------

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
    )

# Remove the original prompt tokens from the generated output
input_length = inputs["input_ids"].shape[1]

output = processor.batch_decode(
    generated_ids[:, input_length:],
    skip_special_tokens=True,
)[0]

print("\nQwen output:")
print(output)

Image: FV001_grand_tour.jpg
Image size: (687, 1024)

Qwen output:
```json
{
  "description": "A colorful, abstract depiction featuring overlapping lines and shapes, including circular and geometric forms. Several stylized spacecraft are shown flying through a vibrant cosmic environment.",
  "primary_topic": "spacecraft",
  "secondary_topics": ["futuristic technology", "astronomy"],
  "visible_elements": ["abstract shapes", "overlapping lines", "colorful circles", "spacecraft", "cosmic environment"]
}
```


### 9. Resumable Batch Semantic Annotation

Processes all 64 original corpus images sequentially with automated topic normalisation, error handling, and real-time JSON checkpointing.

**Features:**
- **Resumable Execution:** Checks `semantic_descriptions.json` on Drive and skips previously processed image IDs if re-run.
- **Robust JSON Parsing:** Strips Markdown code fences and parses structured JSON output dynamically.
- **Topic Normalisation:** Automatically maps non-standard VLM output synonyms (e.g. `vehicle` \(\rightarrow\) `transportation`) to the controlled taxonomy.
- **Interpretive Language Auditing:** Flags subjective phrases (e.g. *"suggesting"*, *"evokes"*) for quality control without interrupting execution.
- **Fault-Tolerant Loop:** Saves updated results after every image and continues execution if an individual asset fails.

In [ ]:
# =========================================================
# Batch VLM semantic description
# Resumable, validated, normalised, and saved to Google Drive
# =========================================================

import json
import re
import time
from pathlib import Path
from PIL import Image

# ---------------------------------------------------------
# Output location
# ---------------------------------------------------------

SEMANTIC_FILE = (
    Path("/content/drive/MyDrive/MOTH Hackathon 2026 Tools")
    / "semantic_descriptions.json"
)

# ---------------------------------------------------------
# Controlled vocabulary
# ---------------------------------------------------------

CONTROLLED_TOPICS = {
    "spacecraft",
    "space station",
    "planetary surface",
    "extraterrestrial habitat",
    "city / settlement",
    "transportation",
    "astronomy",
    "communications",
    "energy / infrastructure",
    "robotics / automation",
    "human activity",
    "futuristic technology",
    "other",
}

# ---------------------------------------------------------
# Topic normalisation
# ---------------------------------------------------------
# Qwen occasionally produces a descriptive synonym that is
# not part of the controlled vocabulary. Map these to the
# nearest approved category rather than expanding the
# vocabulary.

TOPIC_NORMALISATION = {
    "landscape": "planetary surface",
    "planetary landscape": "planetary surface",
    "martian landscape": "planetary surface",
    "lunar landscape": "planetary surface",
    "moon surface": "planetary surface",

    "planet": "astronomy",
    "planets": "astronomy",
    "space": "astronomy",
    "celestial bodies": "astronomy",
    "celestial body": "astronomy",

    "technology": "futuristic technology",
    "future technology": "futuristic technology",

    "robot": "robotics / automation",
    "robots": "robotics / automation",
    "robotics": "robotics / automation",

    "habitat": "extraterrestrial habitat",
    "space habitat": "extraterrestrial habitat",

    "settlement": "city / settlement",
    "city": "city / settlement",
    "cities": "city / settlement",

    "vehicle": "transportation",
    "vehicles": "transportation",
    "transport": "transportation",
    "travel": "transportation",

    "communication": "communications",

    "energy": "energy / infrastructure",
    "infrastructure": "energy / infrastructure",

    "human": "human activity",
    "people": "human activity",
}

# ---------------------------------------------------------
# Phrases to flag for review
# ---------------------------------------------------------
# These do not automatically invalidate a description.
# They are reported so we can inspect the final dataset.

INTERPRETIVE_PHRASES = [
    "suggesting",
    "suggests",
    "suggest",
    "appears to represent",
    "appears to depict",
    "appears to show",
    "evokes",
    "evoking",
    "conveys",
    "conveying",
    "creates a sense of",
    "giving a sense of",
    "symbolises",
    "symbolizes",
    "represents",
    "representing",
    "implies",
    "implying",
    "seems to represent",
    "seems to depict",
]

# ---------------------------------------------------------
# Load existing progress if available
# ---------------------------------------------------------

if SEMANTIC_FILE.exists():

    with open(SEMANTIC_FILE, "r", encoding="utf-8") as f:
        existing_results = json.load(f)

    print(f"Existing results found: {len(existing_results)}")

else:

    existing_results = []

    print("No existing results found. Starting from zero.")


# ---------------------------------------------------------
# Convert existing results into a lookup
# ---------------------------------------------------------
# This prevents duplicate records if a result is regenerated.

semantic_results_by_id = {}

for item in existing_results:

    if "image_id" in item:
        semantic_results_by_id[item["image_id"]] = item


completed_ids = set(semantic_results_by_id.keys())

print(f"Completed image IDs: {len(completed_ids)}")


# ---------------------------------------------------------
# Find the 64 original images
# ---------------------------------------------------------

image_files = sorted(
    ORIGINAL_DIR.glob("*"),
    key=lambda p: p.name
)

image_files = [
    p
    for p in image_files
    if p.suffix.lower() in {
        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
    }
]

print(f"Original image files found: {len(image_files)}")

if len(image_files) != 64:

    raise ValueError(
        f"Expected exactly 64 original images, "
        f"found {len(image_files)}."
    )


# ---------------------------------------------------------
# Helper: extract image ID
# ---------------------------------------------------------

def get_image_id(filename):
    """
    Extract FV001, FV002, etc. from the filename.
    """

    match = re.match(r"(FV\d{3})", filename)

    if not match:

        raise ValueError(
            f"Could not extract image ID from filename: {filename}"
        )

    return match.group(1)


# ---------------------------------------------------------
# Helper: parse model JSON
# ---------------------------------------------------------

def parse_json_output(output):
    """
    Parse JSON returned by Qwen.

    Handles:
    1. raw JSON
    2. JSON wrapped in ```json ... ```
    3. accidental text before/after the JSON object
    """

    output = output.strip()

    # Remove Markdown code fences
    output = re.sub(
        r"^```json\s*",
        "",
        output,
        flags=re.IGNORECASE
    )

    output = re.sub(
        r"\s*```$",
        "",
        output
    )

    output = output.strip()

    # First attempt: direct JSON parsing
    try:

        return json.loads(output)

    except json.JSONDecodeError:

        pass

    # Second attempt: extract the outermost JSON object
    start = output.find("{")
    end = output.rfind("}")

    if start == -1 or end == -1 or end <= start:

        raise ValueError(
            "Could not locate a JSON object in model output."
        )

    json_text = output[start:end + 1]

    return json.loads(json_text)


# ---------------------------------------------------------
# Helper: normalise topic
# ---------------------------------------------------------

def normalise_topic(topic):

    if not isinstance(topic, str):

        raise ValueError(
            f"Topic must be a string, got: {topic!r}"
        )

    topic = topic.strip()

    # Already valid
    if topic in CONTROLLED_TOPICS:
        return topic, False

    # Case-insensitive lookup
    topic_lower = topic.lower()

    for valid_topic in CONTROLLED_TOPICS:

        if topic_lower == valid_topic.lower():

            return valid_topic, True

    # Known synonym
    if topic_lower in TOPIC_NORMALISATION:

        normalised = TOPIC_NORMALISATION[topic_lower]

        return normalised, True

    raise ValueError(
        f"Invalid topic with no normalisation rule: {topic!r}"
    )


# ---------------------------------------------------------
# Helper: normalise semantic result
# ---------------------------------------------------------

def normalise_result(result):

    result = dict(result)

    normalisation_notes = []

    # -----------------------------------------------------
    # Primary topic
    # -----------------------------------------------------

    primary_topic, changed = normalise_topic(
        result["primary_topic"]
    )

    if changed:

        normalisation_notes.append(
            f"primary_topic: "
            f"{result['primary_topic']!r} -> {primary_topic!r}"
        )

    result["primary_topic"] = primary_topic

    # -----------------------------------------------------
    # Secondary topics
    # -----------------------------------------------------

    normalised_secondary = []

    for topic in result["secondary_topics"]:

        normalised_topic, changed = normalise_topic(topic)

        if changed:

            normalisation_notes.append(
                f"secondary_topic: "
                f"{topic!r} -> {normalised_topic!r}"
            )

        if normalised_topic not in normalised_secondary:

            normalised_secondary.append(normalised_topic)

    result["secondary_topics"] = normalised_secondary

    return result, normalisation_notes


# ---------------------------------------------------------
# Helper: validate semantic result
# ---------------------------------------------------------

def validate_result(result):

    required_fields = {
        "description",
        "primary_topic",
        "secondary_topics",
        "visible_elements",
    }

    missing = required_fields - set(result.keys())

    if missing:

        raise ValueError(
            f"Missing fields: {sorted(missing)}"
        )

    # -----------------------------------------------------
    # Primary topic
    # -----------------------------------------------------

    if result["primary_topic"] not in CONTROLLED_TOPICS:

        raise ValueError(
            f"Invalid primary topic: "
            f"{result['primary_topic']}"
        )

    # -----------------------------------------------------
    # Secondary topics
    # -----------------------------------------------------

    if not isinstance(
        result["secondary_topics"],
        list
    ):

        raise ValueError(
            "secondary_topics must be a list."
        )

    invalid_secondary = [
        topic
        for topic in result["secondary_topics"]
        if topic not in CONTROLLED_TOPICS
    ]

    if invalid_secondary:

        raise ValueError(
            f"Invalid secondary topics: "
            f"{invalid_secondary}"
        )

    # -----------------------------------------------------
    # Visible elements
    # -----------------------------------------------------

    if not isinstance(
        result["visible_elements"],
        list
    ):

        raise ValueError(
            "visible_elements must be a list."
        )

    # -----------------------------------------------------
    # Description
    # -----------------------------------------------------

    if not isinstance(
        result["description"],
        str
    ):

        raise ValueError(
            "description must be a string."
        )

    if not result["description"].strip():

        raise ValueError(
            "description is empty."
        )

    return True


# ---------------------------------------------------------
# Helper: detect interpretive wording
# ---------------------------------------------------------

def find_interpretive_phrases(description):

    description_lower = description.lower()

    found = [
        phrase
        for phrase in INTERPRETIVE_PHRASES
        if phrase in description_lower
    ]

    return found


# ---------------------------------------------------------
# Helper: save progress
# ---------------------------------------------------------

def save_results():

    # Save in image ID order
    ordered_results = sorted(
        semantic_results_by_id.values(),
        key=lambda item: item["image_id"]
    )

    with open(
        SEMANTIC_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            ordered_results,
            f,
            indent=2,
            ensure_ascii=False
        )


# ---------------------------------------------------------
# Batch processing
# ---------------------------------------------------------

MAX_IMAGE_SIZE = 1024

remaining = [
    p
    for p in image_files
    if get_image_id(p.name) not in completed_ids
]

print(f"Images remaining: {len(remaining)}")


# ---------------------------------------------------------
# Track warnings
# ---------------------------------------------------------

normalisation_warnings = []
interpretive_warnings = []


# ---------------------------------------------------------
# Run batch
# ---------------------------------------------------------

if not remaining:

    print("✓ All 64 images have already been processed.")

else:

    for index, image_file in enumerate(
        remaining,
        start=1
    ):

        image_id = get_image_id(
            image_file.name
        )

        print()
        print("=" * 70)
        print(
            f"Processing {index}/{len(remaining)}: "
            f"{image_file.name}"
        )
        print("=" * 70)

        try:

            # -------------------------------------------------
            # Load complete original image
            # -------------------------------------------------

            image = Image.open(
                image_file
            ).convert("RGB")

            original_size = image.size

            # Resize complete image without cropping
            image.thumbnail(
                (
                    MAX_IMAGE_SIZE,
                    MAX_IMAGE_SIZE
                ),
                Image.Resampling.LANCZOS
            )

            print(
                f"Original size: {original_size}"
            )

            print(
                f"Model image size: {image.size}"
            )

            # -------------------------------------------------
            # Build Qwen-VL message
            # -------------------------------------------------

            messages = [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "image": image
                        },
                        {
                            "type": "text",
                            "text": VLM_PROMPT
                        },
                    ],
                }
            ]

            text = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

            # -------------------------------------------------
            # Process image + prompt
            # -------------------------------------------------

            inputs = processor(
                text=[text],
                images=[image],
                padding=True,
                return_tensors="pt",
            )

            inputs = {
                k: v.to(model.device)
                if hasattr(v, "to")
                else v
                for k, v in inputs.items()
            }

            # -------------------------------------------------
            # Generate
            # -------------------------------------------------

            with torch.inference_mode():

                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=300,
                    do_sample=False,
                )

            # -------------------------------------------------
            # Remove prompt tokens
            # -------------------------------------------------

            input_length = (
                inputs["input_ids"].shape[1]
            )

            output = processor.batch_decode(
                generated_ids[:, input_length:],
                skip_special_tokens=True,
            )[0]

            # -------------------------------------------------
            # Parse JSON
            # -------------------------------------------------

            result = parse_json_output(
                output
            )

            # -------------------------------------------------
            # Check required structure
            # -------------------------------------------------

            required_fields = {
                "description",
                "primary_topic",
                "secondary_topics",
                "visible_elements",
            }

            missing = (
                required_fields
                - set(result.keys())
            )

            if missing:

                raise ValueError(
                    f"Missing fields: {sorted(missing)}"
                )

            # -------------------------------------------------
            # Normalise topics
            # -------------------------------------------------

            result, notes = normalise_result(
                result
            )

            if notes:

                normalisation_warnings.append(
                    {
                        "image_id": image_id,
                        "notes": notes,
                    }
                )

                print(
                    "Topic normalisation:"
                )

                for note in notes:

                    print(f"  - {note}")

            # -------------------------------------------------
            # Validate after normalisation
            # -------------------------------------------------

            validate_result(result)

            # -------------------------------------------------
            # Check description for interpretive wording
            # -------------------------------------------------

            found_phrases = (
                find_interpretive_phrases(
                    result["description"]
                )
            )

            if found_phrases:

                interpretive_warnings.append(
                    {
                        "image_id": image_id,
                        "phrases": found_phrases,
                    }
                )

                print(
                    "⚠ Interpretive wording detected:"
                )

                print(
                    f"  {found_phrases}"
                )

            # -------------------------------------------------
            # Build stable result record
            # -------------------------------------------------

            result_record = {
                "image_id": image_id,
                "original_filename": image_file.name,
                "description": result["description"],
                "primary_topic": result["primary_topic"],
                "secondary_topics": result["secondary_topics"],
                "visible_elements": result["visible_elements"],
            }

            # -------------------------------------------------
            # Save / update result
            # -------------------------------------------------

            semantic_results_by_id[
                image_id
            ] = result_record

            save_results()

            completed_ids.add(
                image_id
            )

            print("✓ Valid result")

            print(
                f"Primary topic: "
                f"{result['primary_topic']}"
            )

            print(
                f"Description: "
                f"{result['description']}"
            )

            print(
                f"Saved: {SEMANTIC_FILE}"
            )

        except Exception as e:

            print(
                f"✗ ERROR processing "
                f"{image_file.name}"
            )

            print(
                f"{type(e).__name__}: {e}"
            )

            # Continue to the next image
            continue

        # Small pause between images
        time.sleep(0.5)


# ---------------------------------------------------------
# Final summary
# ---------------------------------------------------------

print()
print("=" * 70)
print("BATCH COMPLETE")
print("=" * 70)

final_count = len(
    semantic_results_by_id
)

print(
    f"Results saved: {final_count}"
)

print(
    f"Output file: {SEMANTIC_FILE}"
)

# ---------------------------------------------------------
# Completion status
# ---------------------------------------------------------

if final_count == 64:

    print(
        "✓ All 64 images have semantic descriptions."
    )

else:

    print(
        f"⚠ {64 - final_count} images still need processing."
    )


# ---------------------------------------------------------
# Normalisation summary
# ---------------------------------------------------------

print()
print(
    f"Topic normalisations: "
    f"{len(normalisation_warnings)}"
)

if normalisation_warnings:

    for warning in normalisation_warnings:

        print(
            f"  {warning['image_id']}: "
            f"{'; '.join(warning['notes'])}"
        )


# ---------------------------------------------------------
# Interpretive-language summary
# ---------------------------------------------------------

print()
print(
    f"Interpretive-language warnings: "
    f"{len(interpretive_warnings)}"
)

if interpretive_warnings:

    for warning in interpretive_warnings:

        print(
            f"  {warning['image_id']}: "
            f"{', '.join(warning['phrases'])}"
        )

print()
print(
    "✓ Batch processing cell finished."
)

Existing results found: 11
Completed image IDs: 11
Original image files found: 64
Images remaining: 53

Processing 1/53: FV010_51pegasib.jpg
Original size: (1200, 1789)
Model image size: (687, 1024)
Topic normalisation:
  - secondary_topic: 'travel' -> 'transportation'
✓ Valid result
Primary topic: futuristic technology
Description: A collage of colorful, retro-futuristic travel posters featuring exoplanets like '51 Pegasi b' and 'HD 114762 b'. The posters include vibrant designs with space-themed imagery and text promoting interstellar travel.
Saved: /content/drive/MyDrive/MOTH Hackathon 2026 Tools/semantic_descriptions.json

Processing 2/53: FV013_kepler186f.jpg
Original size: (1200, 1789)
Model image size: (687, 1024)
✓ Valid result
Primary topic: extraterrestrial habitat
Description: Two figures in futuristic suits stand near a white picket fence, pointing towards a forested area with red foliage. The scene is set against a backdrop of a green sky and distant mountains.
Saved: /con

### 10. Exception Handling and Vocabulary Rule Patching

Resolves edge-case processing failures (such as `FV038`) by extending normalisation rules and executing targeted single-asset inference.

**Features:**
- **Synonym Extension:** Adds custom mapping rules (e.g. `medical practice` \(\rightarrow\) `human activity`) to the normalisation dictionary.
- **Vision Processing:** Processes visual inputs via `qwen_vl_utils.process_vision_info`.
- **Dataset Completion:** Updates `semantic_results_by_id` and saves the updated 64-asset dataset.

In [ ]:
# Process the single missing image: FV038

from qwen_vl_utils import process_vision_info

print("✓ process_vision_info imported")

image_file = Path(
    "/content/drive/MyDrive/MOTH Hackathon 2026 Tools/original/"
    "FV038_A_periscope_being_used_above_an_operation_which_is_projected_Wellcome_L0014356.jpg"
)

image_id = "FV038"

print("=" * 70)
print(f"Processing missing image: {image_id}")
print("=" * 70)

# ------------------------------------------------------------------
# Add the FV038 topic normalisation rule
# ------------------------------------------------------------------

TOPIC_NORMALISATION["medical practice"] = "human activity"

print("✓ Added topic normalisation:")
print("  medical practice -> human activity")

# ------------------------------------------------------------------
# Load and resize the full original image
# ------------------------------------------------------------------

image = Image.open(image_file).convert("RGB")

print("Original size:", image.size)

MAX_IMAGE_SIZE = 1024
image.thumbnail(
    (MAX_IMAGE_SIZE, MAX_IMAGE_SIZE),
    Image.Resampling.LANCZOS
)

print("Model image size:", image.size)

# ------------------------------------------------------------------
# Prepare the model input
# ------------------------------------------------------------------

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": VLM_PROMPT},
        ],
    }
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

image_inputs, video_inputs = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)

inputs = {
    k: v.to(model.device) if hasattr(v, "to") else v
    for k, v in inputs.items()
}

# ------------------------------------------------------------------
# Generate
# ------------------------------------------------------------------

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
    )

input_length = inputs["input_ids"].shape[1]

generated_ids_trimmed = [
    out_ids[input_length:]
    for out_ids in generated_ids
]

output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0]

print("\nRaw model output:")
print(output_text)

# ------------------------------------------------------------------
# Parse JSON
# ------------------------------------------------------------------

result = parse_json_output(output_text)

# ------------------------------------------------------------------
# Normalise
# normalise_result() returns:
#     (normalised_result, normalisation_notes)
# ------------------------------------------------------------------

result, normalisation_notes = normalise_result(result)

if normalisation_notes:
    print("\nTopic normalisation:")
    for note in normalisation_notes:
        print("  -", note)

# ------------------------------------------------------------------
# Remove duplicate secondary topics after normalisation
# ------------------------------------------------------------------

result["secondary_topics"] = list(
    dict.fromkeys(result["secondary_topics"])
)

# ------------------------------------------------------------------
# Validate
# ------------------------------------------------------------------

validate_result(result)

# ------------------------------------------------------------------
# Check interpretive wording without rejecting the result
# ------------------------------------------------------------------

warnings = find_interpretive_phrases(
    result["description"]
)

if warnings:
    print("\n⚠ Interpretive wording detected:")
    print(warnings)

# ------------------------------------------------------------------
# Display validated result
# ------------------------------------------------------------------

print("\n✓ Valid result")
print("Primary topic:", result["primary_topic"])
print("Secondary topics:", result["secondary_topics"])
print("Description:", result["description"])

# ------------------------------------------------------------------
# Add stable identifiers
# ------------------------------------------------------------------

result["image_id"] = image_id
result["original_filename"] = image_file.name

# ------------------------------------------------------------------
# Store and save
# ------------------------------------------------------------------

semantic_results_by_id[image_id] = result

save_results()

print("\n✓ FV038 saved")
print(
    "Total semantic results:",
    len(semantic_results_by_id)
)

### 11. Master Semantic Registry Compilation and Export

Sorts the complete 64-asset semantic dataset by ID and exports the final formatted JSON registry to Google Drive.

**Features:**
- **Sequential Ordering:** Organises all records numerically by `image_id` (`FV001` to `FV064`).
- **File Export:** Writes formatted JSON with 2-space indentation to `semantic_descriptions.json`.
- **Status Reporting:** Outputs final completion metrics confirming 100% corpus coverage.

In [ ]:
from pathlib import Path
import json

SEMANTIC_OUTPUT_FILE = (
    Path("/content/drive/MyDrive/MOTH Hackathon 2026 Tools")
    / "semantic_descriptions.json"
)

# Save the complete semantic results
ordered_results = [
    semantic_results_by_id[image_id]
    for image_id in sorted(semantic_results_by_id.keys())
]

with open(SEMANTIC_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(ordered_results, f, indent=2, ensure_ascii=False)

print("=" * 70)
print("SEMANTIC ANNOTATION COMPLETE")
print("=" * 70)
print(f"Results saved: {len(ordered_results)}")
print(f"Output: {SEMANTIC_OUTPUT_FILE}")
print("✓ All 64 images have semantic annotations.")

SEMANTIC ANNOTATION COMPLETE
Results saved: 64
Output: /content/drive/MyDrive/MOTH Hackathon 2026 Tools/semantic_descriptions.json
✓ All 64 images have semantic annotations.


### 12. Local Client File Export

Triggers an automated browser download to transfer the completed semantic metadata registry from Google Drive to your local machine.

**Features:**
- **Colab File Transfer:** Utilises `google.colab.files.download()` to bridge the cloud environment and local client storage.
- **Local Asset Backup:** Downloads the verified `semantic_descriptions.json` file for offline inspection or downstream integration.

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/MOTH Hackathon 2026 Tools/semantic_descriptions.json"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>